In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 0) Load and preprocess
path = "superstore_dataset.csv"  # place the CSV in the same folder as this notebook
df = pd.read_csv(path)

# Basic cleaning
df = df.drop_duplicates().copy()
for col in ["Sales", "Profit", "Discount", "Quantity"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
for dcol in ["Order Date", "Ship Date"]:
    if dcol in df.columns:
        df[dcol] = pd.to_datetime(df[dcol], errors="coerce")

# Keep rows with core fields
core_cols = ["State", "City", "Customer Name", "Category", "Sales", "Profit"]
df = df.dropna(subset=[c for c in core_cols if c in df.columns])

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# 1) Which states have the most sales?
state_sales = df.groupby("State", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
print("\nTop 10 states by total Sales:")
print(state_sales.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=state_sales.head(20), x="Sales", y="State")
plt.title("Top 20 States by Sales")
plt.tight_layout()
plt.show()

# 2) New York vs California: sales and profit
ny = df[df["State"] == "New York"]
ca = df[df["State"] == "California"]

ny_sales = ny["Sales"].sum()
ny_profit = ny["Profit"].sum()
ca_sales = ca["Sales"].sum()
ca_profit = ca["Profit"].sum()

print("\nNY vs CA (Totals):")
print(f"New York - Sales: ${ny_sales:,.0f}, Profit: ${ny_profit:,.0f}")
print(f"California - Sales: ${ca_sales:,.0f}, Profit: ${ca_profit:,.0f}")

plt.figure(figsize=(6,4))
comp = pd.DataFrame({
    "State": ["New York","California"],
    "Sales": [ny_sales, ca_sales],
    "Profit": [ny_profit, ca_profit]
})
comp_m = comp.melt(id_vars="State", value_vars=["Sales","Profit"], var_name="Metric", value_name="Amount")
sns.barplot(data=comp_m, x="Metric", y="Amount", hue="State")
plt.title("New York vs California: Sales and Profit")
plt.tight_layout()
plt.show()

# 3) Outstanding customer in New York (by Profit; also show by Sales)
top_ny_profit = ny.groupby("Customer Name")["Profit"].sum().sort_values(ascending=False).head(1)
top_ny_sales = ny.groupby("Customer Name")["Sales"].sum().sort_values(ascending=False).head(1)
print("\nOutstanding customer in New York (by Profit):")
print(top_ny_profit)
print("\nOutstanding customer in New York (by Sales):")
print(top_ny_sales)

# 4) Differences among states in profitability (profit margin)
state_perf = df.groupby("State", as_index=False)[["Sales","Profit"]].sum()
state_perf["Profit_Margin_%"] = np.where(state_perf["Sales"]>0, state_perf["Profit"]/state_perf["Sales"]*100, np.nan)
state_perf = state_perf.sort_values("Profit_Margin_%", ascending=False)

print("\nStates by Profit Margin (%):")
print(state_perf.head(10))

plt.figure(figsize=(10, 7))
sns.barplot(data=state_perf, x="Profit_Margin_%", y="State")
plt.title("Profitability by State (Profit Margin %)")
plt.tight_layout()
plt.show()

# 5) Pareto principle on customers and Profit (do 20% of customers contribute ~80% of profit?)
cust_profit = df.groupby("Customer Name", as_index=False)["Profit"].sum().sort_values("Profit", ascending=False)
cust_profit["CumProfit"] = cust_profit["Profit"].cumsum()
total_profit = cust_profit["Profit"].sum()
cust_profit["CumShare"] = cust_profit["CumProfit"] / total_profit

n_customers = len(cust_profit)
top_20pct_count = max(1, int(np.ceil(0.20 * n_customers)))
top_20pct_profit_share = cust_profit["Profit"].iloc[:top_20pct_count].sum() / total_profit * 100

print(f"\nPareto (Profit): top 20% customers count = {top_20pct_count} of {n_customers}")
print(f"Share of total profit by top 20% customers: {top_20pct_profit_share:.1f}%")

plt.figure(figsize=(8,4))
plt.plot(np.arange(1, n_customers+1)/n_customers*100, cust_profit["CumShare"]*100)
plt.axvline(x=20, linestyle="--")
plt.axhline(y=80, linestyle="--")
plt.xlabel("Customers (cumulative %)")
plt.ylabel("Profit (cumulative %)")
plt.title("Cumulative Profit by Customers (Pareto Curve)")
plt.tight_layout()
plt.show()

# 6) Top 20 cities by Sales and by Profit; differences in profitability
city_perf = df.groupby("City", as_index=False)[["Sales","Profit"]].sum()
city_perf["Profit_Margin_%"] = np.where(city_perf["Sales"]>0, city_perf["Profit"]/city_perf["Sales"]*100, np.nan)

top20_sales_cities = city_perf.sort_values("Sales", ascending=False).head(20)
top20_profit_cities = city_perf.sort_values("Profit", ascending=False).head(20)

print("\nTop 20 cities by Sales:")
print(top20_sales_cities[["City","Sales","Profit","Profit_Margin_%"]])
print("\nTop 20 cities by Profit:")
print(top20_profit_cities[["City","Sales","Profit","Profit_Margin_%"]])

plt.figure(figsize=(10,6))
sns.barplot(data=top20_sales_cities, x="Sales", y="City")
plt.title("Top 20 Cities by Sales")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,6))
sns.barplot(data=top20_profit_cities, x="Profit", y="City")
plt.title("Top 20 Cities by Profit")
plt.tight_layout()
plt.show()

# 7) Top 20 customers by Sales
cust_sales = df.groupby("Customer Name", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False).head(20)
print("\nTop 20 customers by Sales:")
print(cust_sales)

plt.figure(figsize=(10,6))
sns.barplot(data=cust_sales, x="Sales", y="Customer Name")
plt.title("Top 20 Customers by Sales")
plt.tight_layout()
plt.show()

# 8) Cumulative curve in Sales by Customers (Pareto for Sales)
cust_sales_all = df.groupby("Customer Name", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
cust_sales_all["CumSales"] = cust_sales_all["Sales"].cumsum()
total_sales = cust_sales_all["Sales"].sum()
cust_sales_all["CumShare"] = cust_sales_all["CumSales"]/total_sales*100

n_cust = len(cust_sales_all)
top_20pct_sales_share = cust_sales_all["Sales"].iloc[:max(1, int(np.ceil(0.20*n_cust)))].sum() / total_sales * 100
print(f"\nPareto (Sales): top 20% customers share = {top_20pct_sales_share:.1f}%")

plt.figure(figsize=(8,4))
plt.plot(np.arange(1, n_cust+1)/n_cust*100, cust_sales_all["CumShare"])
plt.axvline(x=20, linestyle="--")
plt.axhline(y=80, linestyle="--")
plt.xlabel("Customers (cumulative %)")
plt.ylabel("Sales (cumulative %)")
plt.title("Cumulative Sales by Customers (Pareto Curve)")
plt.tight_layout()
plt.show()

# 9) Simple recommendations based on analysis
# Prioritize states and cities with high profit and strong margins
top_states_by_profit = state_perf.sort_values("Profit", ascending=False) if "Profit" in state_perf.columns else state_perf
top_states = top_states_by_profit.head(10)[["State","Sales","Profit","Profit_Margin_%"]]
profitable_cities = city_perf.sort_values(["Profit","Profit_Margin_%"], ascending=[False, False]).head(20)

print("\nRecommendations:")
print("- States to prioritize (high profit and margin):")
print(top_states.to_string(index=False))
print("\n- Cities to prioritize (high profit and margin):")
print(profitable_cities[["City","Sales","Profit","Profit_Margin_%"]].to_string(index=False))
